<a href="https://colab.research.google.com/github/Koshalendar/Artificial-intelligence-/blob/main/ant_colony_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random

num_exams = int(input("Enter number of exams: "))
num_slots = int(input("Enter number of time slots: "))
num_rooms = int(input("Enter number of rooms: "))

enrollments = {}
for i in range(num_exams):
    enrollments[i] = int(input(f"Enter enrollment size for Exam {i}: "))

room_capacities = {}
for i in range(num_rooms):
    room_capacities[i] = int(input(f"Enter capacity for Room {i}: "))

conflict_matrix = {}
print("Enter student conflicts between exams (count of common students):")
for i in range(num_exams):
    for j in range(i + 1, num_exams):
        c = int(input(f"Conflicts between Exam {i} and Exam {j}: "))
        conflict_matrix[(i, j)] = c
        conflict_matrix[(j, i)] = c

alpha = 1.0
beta = 2.0
rho = 0.1
Q = 100.0
num_ants = 5
max_iterations = 20

pheromone = {}
for e in range(num_exams):
    for s in range(num_slots):
        for r in range(num_rooms):
            pheromone[(e, s, r)] = 1.0

global_best_timetable = None
global_best_cost = float('inf')

for iteration in range(max_iterations):
    ants_timetables = []
    ants_costs = []

    for ant in range(num_ants):
        timetable = {}
        assigned_exams = set()
        success = True

        exam_order = sorted(range(num_exams), key=lambda x: sum(conflict_matrix.get((x, y), 0) for y in range(num_exams)), reverse=True)

        for e in exam_order:
            valid_options = []

            for s in range(num_slots):
                for r in range(num_rooms):
                    if enrollments[e] > room_capacities[r]:
                        continue

                    room_busy = False
                    student_conflict = False
                    for prev_e, (prev_s, prev_r) in timetable.items():
                        if prev_s == s:
                            if prev_r == r:
                                room_busy = True
                                break
                            if conflict_matrix.get((e, prev_e), 0) > 0:
                                student_conflict = True
                                break

                    if not room_busy and not student_conflict:
                        valid_options.append((s, r))

            if not valid_options:
                success = False
                break

            weights = []
            for s, r in valid_options:
                soft_penalties = 0
                for prev_e, (prev_s, prev_r) in timetable.items():
                    if abs(prev_s - s) == 1:
                        soft_penalties += conflict_matrix.get((e, prev_e), 0)

                eta = 1.0 / (1.0 + soft_penalties)
                tau = pheromone[(e, s, r)]
                weight = (tau ** alpha) * (eta ** beta)
                weights.append(weight)

            chosen_idx = random.choices(range(len(valid_options)), weights=weights, k=1)[0]
            timetable[e] = valid_options[chosen_idx]

        if success:
            total_cost = 0
            for e1 in range(num_exams):
                for e2 in range(e1 + 1, num_exams):
                    s1, _ = timetable[e1]
                    s2, _ = timetable[e2]
                    if abs(s1 - s2) == 1:
                        total_cost += conflict_matrix.get((e1, e2), 0)

            ants_timetables.append(timetable)
            ants_costs.append(total_cost)

            if total_cost < global_best_cost:
                global_best_cost = total_cost
                global_best_timetable = timetable.copy()

    for key in pheromone:
        pheromone[key] *= (1.0 - rho)

    if global_best_timetable:
        for e, (s, r) in global_best_timetable.items():
            pheromone[(e, s, r)] += (Q / (1.0 + global_best_cost))

print("\n--- Best Timetable Found ---")
if global_best_timetable:
    print(f"Total Penalty Cost: {global_best_cost}")
    for e in sorted(global_best_timetable.keys()):
        s, r = global_best_timetable[e]
        print(f"Exam {e} -> Time Slot {s}, Room {r}")
else:
    print("No valid timetable could be found satisfying the hard constraints.")



Enter number of exams: 3
Enter number of time slots: 2
Enter number of rooms: 2
Enter enrollment size for Exam 0: 50
Enter enrollment size for Exam 1: 30
Enter enrollment size for Exam 2: 40
Enter capacity for Room 0: 60
Enter capacity for Room 1: 35
Enter student conflicts between exams (count of common students):
Conflicts between Exam 0 and Exam 1: 0
Conflicts between Exam 0 and Exam 2: 12
Conflicts between Exam 1 and Exam 2: 5

--- Best Timetable Found ---
Total Penalty Cost: 17
Exam 0 -> Time Slot 1, Room 0
Exam 1 -> Time Slot 1, Room 1
Exam 2 -> Time Slot 0, Room 0
